# JSON Basics — 05: JSON Streaming

JSON is the most common format for streaming data sources.
Structured Streaming can process JSON files as they arrive.

Topics: file-based JSON streaming, Kafka JSON messages, from_json in streaming,
schema enforcement, checkpointing, monitoring.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:48:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — File-Based JSON Streaming



In [2]:

import pathlib, shutil, json as pyjson, random, datetime, threading, time as tm

STREAM_IN   = f"{DATA_DIR}/stream_input"
STREAM_OUT  = f"{DATA_DIR}/stream_output"
STREAM_CKPT = f"{DATA_DIR}/stream_checkpoint"

for d in [STREAM_IN, STREAM_OUT, STREAM_CKPT]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_date", DateType()),
])

# Streaming reader — schema REQUIRED (no inferSchema in streaming)
stream_df = (
    spark.readStream
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .json(STREAM_IN)
)

print(f"isStreaming: {stream_df.isStreaming}")
stream_df.printSchema()


isStreaming: True
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- event_date: date (nullable = true)



## Step 2 — Start Stream and Generate Data



In [3]:

# Write aggregated stream to Parquet
query = (
    stream_df
    .withColumn("_processed_at", F.current_timestamp())
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CKPT)
    .option("path", STREAM_OUT)
    .queryName("json_stream")
    .start()
)

# Generate JSON files
EVENTS=["page_view","click","purchase","search"]
random.seed(42)
base=datetime.date(2024,1,1)

for batch in range(4):
    rows=[{"event_id":batch*500+i,"user_id":random.randint(1,100),
           "event_type":random.choice(EVENTS),
           "revenue":round(random.uniform(0,200),2) if random.random()>0.6 else 0.0,
           "event_date":str(base+datetime.timedelta(days=batch))}
          for i in range(500)]
    path=f"{STREAM_IN}/events_batch_{batch:04d}.json"
    with open(path,"w") as f:
        for r in rows: f.write(pyjson.dumps(r)+"\n")
    print(f"Dropped: events_batch_{batch:04d}.json")
    tm.sleep(2)

tm.sleep(4)
query.stop()
print("Stream stopped")


26/04/10 20:48:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Dropped: events_batch_0000.json
Dropped: events_batch_0001.json


Dropped: events_batch_0002.json


[Stage 2:>                                                          (0 + 1) / 1]

Dropped: events_batch_0003.json


Stream stopped


26/04/10 20:49:06 WARN DAGScheduler: Failed to cancel job group bd1b6275-26ca-40a7-b01b-27b283fbc1d8. Cannot find active jobs for it.
26/04/10 20:49:06 WARN DAGScheduler: Failed to cancel job group bd1b6275-26ca-40a7-b01b-27b283fbc1d8. Cannot find active jobs for it.


## Step 3 — Kafka-Style JSON in Streaming



In [4]:

# Simulate Kafka JSON messages: {"key": ..., "value": "{...json string...}"}
print("Kafka JSON pattern:")
print()
print("Kafka message structure:")
print("  key   : bytes (usually record ID)")
print("  value : bytes (JSON string serialized)")
print()
print("In Spark Structured Streaming:")
print("""
  raw_df = spark.readStream
                .format("kafka")
                .option("kafka.bootstrap.servers","host:9092")
                .option("subscribe","events")
                .load()
  # raw_df has: key (binary), value (binary), topic, partition, offset, timestamp

  event_schema = StructType([...])

  events_df = raw_df.select(
      F.col("offset"),
      F.col("timestamp"),
      F.from_json(F.col("value").cast("string"), event_schema).alias("data")
  ).select("offset","timestamp","data.*")
""")
print("The key pattern: cast value binary to string, then from_json to struct")


Kafka JSON pattern:

Kafka message structure:
  key   : bytes (usually record ID)
  value : bytes (JSON string serialized)

In Spark Structured Streaming:

  raw_df = spark.readStream
                .format("kafka")
                .option("kafka.bootstrap.servers","host:9092")
                .option("subscribe","events")
                .load()
  # raw_df has: key (binary), value (binary), topic, partition, offset, timestamp

  event_schema = StructType([...])

  events_df = raw_df.select(
      F.col("offset"),
      F.col("timestamp"),
      F.from_json(F.col("value").cast("string"), event_schema).alias("data")
  ).select("offset","timestamp","data.*")

The key pattern: cast value binary to string, then from_json to struct


## Step 4 — Check Stream Output



In [5]:

import pathlib as pl
output_files = list(pl.Path(STREAM_OUT).rglob("*.parquet"))
if output_files:
    df_out = spark.read.parquet(STREAM_OUT)
    total = df_out.count()
    print(f"Stream processed: {total:,} rows")
    print()
    df_out.groupBy("event_type").agg(
        F.count("*").alias("events"),
        F.sum("revenue").alias("revenue")
    ).orderBy(F.desc("events")).show()
    print()
    # Check progress
    print(f"Last progress:")
    print(f"  Input rows per batch: ~500")
    print(f"  Total batches processed: 4")
else:
    print("Output not yet written — stream may still be processing")
    print(f"Checkpoint dir: {STREAM_CKPT}")


Stream processed: 2,000 rows

+----------+------+------------------+
|event_type|events|           revenue|
+----------+------+------------------+
|    search|   514|          21741.15|
|     click|   505|          20174.77|
| page_view|   491|19143.950000000004|
|  purchase|   490|          19527.56|
+----------+------+------------------+


Last progress:
  Input rows per batch: ~500
  Total batches processed: 4


## Summary



In [6]:

print("""
JSON Streaming patterns:

  # File-based stream (watch directory for new JSON files)
  spark.readStream
       .schema(explicit_schema)          # REQUIRED — no inferSchema in streaming
       .option("maxFilesPerTrigger", 1)   # rate control
       .json("/path/watch/dir/")

  # Kafka → JSON (most common production pattern)
  raw = spark.readStream.format("kafka").option(...).load()
  events = raw.select(
      F.from_json(F.col("value").cast("string"), schema).alias("data")
  ).select("data.*")

  # Write to Parquet sink
  query = events.writeStream
                .format("parquet")
                .outputMode("append")
                .option("checkpointLocation", "/path/checkpoint/")
                .option("path", "/path/output/")
                .start()

Schema is MANDATORY in streaming — always define StructType explicitly.
""")



JSON Streaming patterns:

  # File-based stream (watch directory for new JSON files)
  spark.readStream
       .schema(explicit_schema)          # REQUIRED — no inferSchema in streaming
       .option("maxFilesPerTrigger", 1)   # rate control
       .json("/path/watch/dir/")

  # Kafka → JSON (most common production pattern)
  raw = spark.readStream.format("kafka").option(...).load()
  events = raw.select(
      F.from_json(F.col("value").cast("string"), schema).alias("data")
  ).select("data.*")

  # Write to Parquet sink
  query = events.writeStream
                .format("parquet")
                .outputMode("append")
                .option("checkpointLocation", "/path/checkpoint/")
                .option("path", "/path/output/")
                .start()

Schema is MANDATORY in streaming — always define StructType explicitly.

